# Lesson 19: Multimodal AI: vision, documents, speech and image generation

*Companion notebook for [the lesson page](../lessons/19-multimodal.html). Worked solutions are in `solutions/19-multimodal-solutions.ipynb`; try each exercise first.*

Extend everything you know beyond text, from vision transformers and CLIP to sending images and documents to vision-language models, transcribing speech, and understanding how diffusion models generate images.

**Runs on:** CPU for most cells; an API key for the vision-language model; a GPU for image generation.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy matplotlib pillow transformers diffusers accelerate anthropic pydantic python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()

## Vision

### CLIP: connecting images and text

- **Zero-shot classification**: embed candidate labels as sentences ("a photo of a corroded connector") and pick the closest to the image. No training needed.
- **Text-to-image search** and **image-to-image search** over photo collections.
- The text-conditioning backbone of many image generators.

In [ ]:
# Uses your own files: edit the paths or data first; Downloads a model or data on first run
# uv add transformers torch pillow
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

image = Image.open("field_photo.jpg")
labels = ["a clean optical window", "a dirty optical window", "a cracked optical window", "a bird nest"]
inputs = proc(text=[f"a photo of {l}" for l in labels], images=image, return_tensors="pt", padding=True)
probs = model(**inputs).logits_per_image.softmax(dim=1)[0]
for label, p in sorted(zip(labels, probs.tolist()), key=lambda x: -x[1]):
    print(f"{p:.2f}  {label}")

### Vision-language models through an API

A **vision-language model (VLM)** connects a vision encoder to an LLM, so images become tokens in the conversation. You send images as content blocks next to text.

In [ ]:
# Make a synthetic spectrum plot to send to the vision-language model
import numpy as np, matplotlib.pyplot as plt
wl = np.linspace(300, 500, 800)
spec = np.exp(-((wl - 480) / 120) ** 2) * (1 - 0.6 * np.exp(-((wl - 393.4) / 1.2) ** 2)
                                          - 0.5 * np.exp(-((wl - 396.8) / 1.2) ** 2))
spec[400:405] += 0.4                       # an artefact spike for the model to notice
plt.figure(figsize=(8, 4)); plt.plot(wl, spec)
plt.xlabel("Wavelength (nm)"); plt.ylabel("Signal (normalised)"); plt.title("Measured spectrum")
plt.savefig("spectrum_plot.png", dpi=120, bbox_inches="tight"); plt.show()

In [ ]:
# Needs an API key (see the setup cell)
import base64, anthropic

client = anthropic.Anthropic()

def ask_about_image(path: str, question: str, media_type: str = "image/jpeg") -> str:
    data = base64.standard_b64encode(open(path, "rb").read()).decode()
    r = client.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=800,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": media_type, "data": data}},
                {"type": "text", "text": question},
            ],
        }],
    )
    return r.content[0].text

print(ask_about_image("spectrum_plot.png",
                      "Describe this plot: axes, units, the main features, and anything that looks like an artefact.",
                      media_type="image/png"))

## Documents and speech

### Document understanding

1. **Text extraction** with PDF libraries for digital PDFs (lesson 13): fast and cheap, but loses layout and fails on scans.
2. **OCR** (Tesseract, cloud OCR services, or layout-aware tools such as Docling) for scanned pages: produces text, sometimes with layout and tables.
3. **Send page images to a VLM** with a schema: best on messy layouts, handwriting, tables and forms, at higher cost per page. Many APIs also accept PDFs directly as document blocks.

In [ ]:
from pydantic import BaseModel

class CalibrationCert(BaseModel):
    instrument_serial: str
    calibration_date: str          # ISO format, YYYY-MM-DD
    lab_name: str
    wavelength_range_nm: tuple[float, float] | None
    passed: bool

# Render each PDF page to PNG (e.g. with PyMuPDF), then for each page image:
# use the forced-tool-call pattern from lesson 10 with input_schema=CalibrationCert.model_json_schema(),
# sending the image block plus "Extract the certificate fields. Use null for anything not present."

### Speech-to-text

**Whisper** (2022) is an encoder-decoder transformer (lesson 08) trained on hundreds of thousands of hours of audio. It transcribes and translates many languages and runs locally in sizes from tiny to large.

In [ ]:
# Uses your own files: edit the paths or data first; Downloads a model or data on first run
# uv add transformers torch  (and install ffmpeg on your system for audio decoding)
from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model="openai/whisper-small")
result = asr("site_visit_notes.m4a", return_timestamps=True)
print(result["text"])

## Generating images: diffusion models

### The intuition

and the network $\varepsilon_\theta(x_t, t)$ is trained with a simple mean-squared error, $\lVert \varepsilon - \varepsilon_\theta(x_t, t)\rVert^2$.

In [ ]:
import numpy as np

T = 1000
betas = np.linspace(1e-4, 0.02, T)              # the linear schedule from the original DDPM paper
alpha_bar = np.cumprod(1 - betas)

rng = np.random.default_rng(0)
x0 = np.sin(np.linspace(0, 4 * np.pi, 200))     # a stand-in "image": a clean signal

def q_sample(x0, t):
    eps = rng.normal(size=x0.shape)
    return np.sqrt(alpha_bar[t]) * x0 + np.sqrt(1 - alpha_bar[t]) * eps, eps

for t in [0, 100, 300, 999]:
    xt, _ = q_sample(x0, t)
    print(t, round(float(np.corrcoef(x0, xt)[0, 1]), 2))
# 0 1.0    100 0.9    300 0.57    999 -0.05: by the last step, nothing of the signal remains

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(14, 2.5), sharey=True)
for ax, t in zip(axes, [0, 100, 300, 999]):
    ax.plot(q_sample(x0, t)[0], lw=0.8)
    ax.set_title(f"t = {t}")
plt.show()

### What makes modern image generators work

In [ ]:
# Needs a GPU runtime (Colab: Runtime > Change runtime type > GPU)
# on a GPU: uv add diffusers transformers accelerate torch
import torch
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", torch_dtype=torch.float16).to("cuda")
image = pipe(prompt="a weatherproof scientific instrument on a mountain ridge at dawn, photo",
             num_inference_steps=1, guidance_scale=0.0).images[0]      # sd-turbo is distilled for 1 to 4 steps
image.save("instrument.png")

### Hands-on: a field-photo triage pipeline

You have a folder of photos from site visits and want to flag ones needing attention.

1. Use CLIP zero-shot with 4 to 6 descriptive labels to triage 30 or more images (your own photos, or public-domain images of equipment, weather and landscapes). Measure accuracy against your own labels.
2. For images CLIP flags as problems, call a VLM with a Pydantic schema: `{issue_type, severity 1-5, visible_evidence, recommended_action}`.
3. Compare cost and time per image for the two stages. Why is the two-stage design sensible?

In [ ]:
# Your code here

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/19-multimodal.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.